# 04 Equity analysis

Each stop takes the values of its neighborhood: median household income, poverty share and share of households without a car.

* **Transit dependency** = 0.4 × poverty share + 0.6 × car-free share
* **Equity impact** = vulnerability score × transit dependency
* **Equity class** from the score and neighborhood income (rules in `src/equity.py`)

`data/reference/` holds the 20 neighborhood values and the stop-to-neighborhood assignment behind the published results. They are coarse, one value per neighborhood; for new work, replace them with ACS tract or block-group estimates joined by location.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
(DATA / "processed").mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
from src import equity

scores = pd.read_csv(DATA / "processed" / "centrality_vulnerability.csv")
areas = pd.read_csv(DATA / "reference" / "neighborhoods.csv")
assign = pd.read_csv(DATA / "reference" / "stop_neighborhoods.csv")

stops = scores.merge(assign, on="stop_id", how="left")
print("Stops without a neighborhood:", stops.neighborhood.isna().sum())
eq = equity.add_equity(stops.dropna(subset=["neighborhood"]), areas)
eq["equity_class"].value_counts()

In [ ]:
by_hood = (eq.groupby("neighborhood")
             .agg(stops=("stop_id", "size"), mean_score=("vulnerability_score", "mean"),
                  median_income=("median_income", "first"), transit_dependency=("transit_dependency", "first"),
                  high_or_critical=("equity_class", lambda c: c.isin(["High Equity Risk", "Critical Equity Risk"]).mean()))
             .sort_values("mean_score", ascending=False).round(3))
by_hood

In [ ]:
eq.sort_values("equity_impact", ascending=False).head(15)[["stop_name", "neighborhood", "vulnerability_score", "transit_dependency", "equity_impact", "equity_class"]]

In [ ]:
eq.to_csv(DATA / "processed" / "stops_with_equity.csv", index=False)
print("Saved stops_with_equity.csv")